---
title: "Лекція 3. Дизайн ML-дослідження та валідність наукових висновків"
description: "Машинне навчання та методи вилучення знань | PhD F3 «Комп'ютерні науки» | КрНУ ім. М. Остроградського"
author: "&copy; [Valeriy Sydorenko](https://www.linkedin.com/in/valeriy-sydorenko-6782279a/), 2026"
date: today
jupyter: python3
bibliography: mlkdd-references.bib

format:
  html:
    code-fold: show
    code-summary: "Показати код"
    code-overflow: wrap
    toc: true
    toc-depth: 4
    toc-location: right
    number-sections: false
    embed-resources: true
    smooth-scroll: true
---

## Мета лекції

Сформувати здатність *проєктувати машинне навчання як науковий експеримент*: від дослідницького питання й протоколу до відтворюваних результатів, аудиту загроз валідності та обґрунтованого наукового твердження. Особлива увага -- відповідності схеми валідації реальному сценарію застосування моделі.

*Очікувані результати.* Після лекції аспірант обґрунтовує baseline, схему розбиття даних, процедуру підбору гіперпараметрів, метрики, оцінювання невизначеності й перевірки стійкості; розрізняє внутрішню, зовнішню, конструктну та статистичну валідність; формулює висновок із явним зазначенням його меж.

*Зв'язок із курсом.* Лекція 2 пояснила, *чому* оцінка якості невизначена й залежить від розподілу даних. Лекція 3 відповідає, *як* організувати дослідження, щоб цю невизначеність не приховати, а врахувати.


## 1. Модель -- лише один компонент ML-дослідження

У науковому дослідженні недостатньо отримати велике значення accuracy, F1 або ROC AUC. Потрібно показати, *що саме оцінюється, на яких даних, за якою процедурою та яке твердження це дозволяє сформулювати* [@breiman2001two; @hastie2009elements].

Розрізняємо чотири рівні:

- *Дослідницьке питання (research question, RQ)* -- що хочемо встановити про явище?
- *Операціоналізація (operationalization)* -- як поняття перетворюються на змінні, мітки, ознаки й метрики?
- *Експериментальний протокол (experimental protocol)* -- як саме отримуємо незалежні свідчення?
- *Наукове твердження (scientific claim)* -- що доводять результати та чого вони не доводять?

Наприклад, RQ «Чи можна виявляти ботоподібну активність на ранніх етапах?» *не тотожне* задачі «максимізувати F1 на випадковому test split». Для раннього виявлення важливі момент доступності ознак, часовий поділ і перевірка на нових облікових записах.


## 2. Доказовий ланцюг ML-експерименту

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart LR
 R("RQ / гіпотеза") --> O("Операціоналізація")
 O --> D("Дані та вибірка")
 D --> P("Протокол")
 P --> E("Експеримент")
 E --> A("Аудит валідності")
 A --> C("Наукове твердження")
```

Кожний перехід може створити загрозу валідності: неправильно виміряний конструкт, нерепрезентативна вибірка, витік даних, довільний вибір метрики або надмірне узагальнення. Відтворюваність коду сама по собі не усуває цих загроз.


## 3. Гіпотеза, оцінювана величина та критерій перевірки

Перед експериментом потрібно сформулювати *оцінювану величину (estimand)*. Для порівняння двох моделей A і B на цільовій популяції це може бути різниця очікуваних втрат:

$$
\Delta = \mathbb{E}_{(X,Y)\sim P_{\mathrm{target}}}\!\left[\ell(f_A(X),Y)-\ell(f_B(X),Y)\right].
$$

Якщо \(\Delta<0\), модель A має меншу очікувану втрату за визначених умов. Однак експеримент дає лише оцінку \(\widehat\Delta\), яка залежить від вибірки, схеми навчання та випадковості.

*Мініприклад.* Якщо на тестовій вибірці середня втрата A дорівнює 0,18, а B -- 0,20, то \(\widehat\Delta=-0{,}02\). Без оцінки невизначеності, аналізу залежностей і відповідності test-популяції цільовій популяції цей результат не підтверджує загальної переваги A.


## 4. Базова модель (baseline) як наукова точка відліку

Baseline -- не «слабка модель для галочки», а контрольна умова, без якої складно інтерпретувати приріст якості [@hastie2009elements].

Для класифікації доцільно розрізняти:

1. *Наївний baseline* -- прогноз найчастішого класу.
2. *Простий змістовний baseline* -- логістична регресія, дерево невеликої глибини або правило предметної області.
3. *Сильний референтний baseline* -- перевірений метод для конкретної постановки.

Порівняння повинно бути чесним: однакова тестова популяція, доступність інформації, метрика, бюджет налаштування та спосіб обчислення невизначеності.

::: {.callout-important}
## Важливо
Перевага над слабким baseline не є доказом наукової новизни. А відсутність приросту над сильним baseline може бути важливим негативним науковим результатом.
:::


## 5. Розбиття даних як частина дослідницької гіпотези

Стандартне випадкове розбиття має сенс лише за відповідних припущень щодо незалежності та однакового розподілу спостережень (i.i.d.). У реальних задачах одиниці спостереження часто пов'язані.

| Сценарій | Коректніша перевірка | Чого не гарантує випадковий split |
|---|---|---|
| Незалежні об'єкти з однієї популяції | Stratified train/test або CV | Узагальнення на інші популяції |
| Кілька записів одного користувача/пацієнта | Group split / GroupKFold | Незалежність між train і test |
| Прогноз у майбутньому | Chronological split / rolling validation | Відсутність інформації з майбутнього |
| Нові установи, регіони, платформи | Leave-one-domain-out / external test | Перенесення на новий домен |

*Одиниця незалежності* може не збігатися з рядком таблиці. Це принципове питання дизайну, а не лише вибір функції scikit-learn.


## 6. Де виникає витік даних (data leakage)

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart TD
 D("Повний набір даних") --> S("Розділення на train/test")
 S --> TR("Train: fit preprocessing + model")
 S --> TE("Test: лише transform + evaluate")
 TR --> M("Заморожена модель")
 M --> TE
 X("ПОМИЛКА: fit preprocessing на всіх даних") -.-> TE
```

Витік даних виникає, коли інформація, яка в реальному сценарії недоступна на момент прогнозу, впливає на навчання, вибір ознак, підбір параметрів або оцінювання. Основні випадки: preprocessing до розбиття, дублікати чи споріднені об'єкти в train/test, використання майбутніх подій, повторне налаштування за test-результатами.

**Коректна практика:** усі трансформації, що навчаються з даних, включають у `Pipeline`; їх параметри оцінюються лише на train-частині відповідного розбиття.


## 7. Навчальний експеримент: випадковий split проти group split

Змоделюємо ситуацію, коли кожний суб'єкт має багато вимірювань. Вимірювання одного суб'єкта подібні, тому випадкове розбиття рядків дозволяє моделі «впізнавати» вже бачених суб'єктів. Це дає завищену оцінку якості на *нових* суб'єктах.

Важливо: це *синтетична демонстрація механізму витоку між групами*, а не універсальна оцінка його величини.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

rng = np.random.default_rng(42)
n_subjects, records_per_subject, n_features = 120, 8, 12
subject_vectors = rng.normal(size=(n_subjects, n_features))
subject_labels = rng.integers(0, 2, size=n_subjects)
X = np.repeat(subject_vectors, records_per_subject, axis=0)
X += rng.normal(scale=0.08, size=X.shape)
y = np.repeat(subject_labels, records_per_subject)
groups = np.repeat(np.arange(n_subjects), records_per_subject)

row_train, row_test = train_test_split(
    np.arange(len(y)), test_size=0.25, random_state=42, stratify=y
)
group_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
group_train, group_test = next(group_split.split(X, y, groups))

def evaluate(train_idx, test_idx):
    model = KNeighborsClassifier(n_neighbors=1)
    model.fit(X[train_idx], y[train_idx])
    return accuracy_score(y[test_idx], model.predict(X[test_idx]))

pd.DataFrame({
    'Схема': ['Випадкове розбиття рядків', 'Розбиття за суб’єктами'],
    'Accuracy': [evaluate(row_train, row_test), evaluate(group_train, group_test)],
    'Спільні суб’єкти train/test': [
        len(set(groups[row_train]) & set(groups[row_test])),
        len(set(groups[group_train]) & set(groups[group_test]))
    ]
}).round(3)


,Схема,Accuracy,Спільні суб’єкти train/test
0,Випадкове розбиття рядків,1.000,102
1,Розбиття за суб’єктами,0.517,0


### Інтерпретація результату

Випадкове розбиття може показати майже ідеальну accuracy, хоча модель не вивчила узагальнюваний зв'язок між ознаками та класом: у синтетичному прикладі мітка призначена суб'єкту випадково. Для нових суб'єктів така закономірність відсутня.

*Контрольне питання:* яка саме популяція відповідає кожному test split? Якщо мета -- класифікувати нові записи *відомих* суб'єктів, одна схема може бути релевантною; якщо мета -- нових суб'єктів, потрібна інша.


## 8. Cross-validation: оцінка протоколу, а не магічна гарантія

У $K$-кратній перехресній перевірці (K-fold cross-validation, CV) набір ділиться на $K$ частин; кожна послідовно стає валідаційною. Середня оцінка:

$$
\widehat{R}_{CV}=\frac{1}{K}\sum_{k=1}^{K}\widehat{R}_k.
$$

Залежно від задачі використовують `StratifiedKFold`, `GroupKFold`, часові або блокові схеми. *Неправильна CV-схема повторює систематичну помилку K разів*, а не виправляє її.

Оцінки між folds залежні, оскільки навчальні набори перекриваються. Стандартне відхилення fold-метрик не слід автоматично інтерпретувати як коректний довірчий інтервал для якості на цільовій популяції.


## 9. Підбір гіперпараметрів і nested cross-validation

Коли гіперпараметри обирають за CV-результатами, найкраща CV-оцінка є оптимістичною оцінкою якості обраної конфігурації. *Вкладена перехресна перевірка (nested CV)* розділяє дві функції: внутрішній цикл вибирає модель, зовнішній оцінює всю процедуру вибору [@hastie2009elements].


```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart TD
 O("Зовнішній train fold") --> I("Внутрішня CV")
 I --> H("Вибір гіперпараметрів")
 H --> F("Fit на зовнішньому train")
 F --> T("Оцінка на зовнішньому test fold")
 T --> A("Агрегація зовнішніх оцінок")
```

Якщо є окремий фінальний test set, його використовують *лише після* фіксації дизайну й вибору моделі. Повторні спроби поліпшити модель після перегляду test-метрики фактично перетворюють test на ще одну валідаційну вибірку.


## 10. Метрика повинна відповідати дослідницькому питанню

Accuracy може бути неінформативною за дисбалансу класів. Для раннього виявлення рідкісних подій можуть бути важливими recall за фіксованого рівня false positives, precision, PR AUC, калібрування й час до виявлення.

Для ймовірнісних прогнозів оцінюємо не тільки дискримінацію, а й калібрування: чи відповідає частота події заявленій імовірності. Для прикладної наукової задачі слід вказати *цільову метрику до перегляду результатів*, а також другорядні показники й критерії прийнятності.

**Приклад:** 98% accuracy за частки позитивного класу 2% досягається прогнозом лише негативного класу. Такий результат не свідчить про здатність виявляти позитивні випадки.


## 11. Невизначеність порівняння моделей

Порівняння A і B на *тих самих* тестових об'єктах утворює парний дизайн. Нехай $d_i=\ell(f_A(x_i),y_i)-\ell(f_B(x_i),y_i)$. Тоді

$$
\widehat\Delta=\frac{1}{n}\sum_{i=1}^{n}d_i.
$$

Варіабельність різниці може бути меншою, ніж варіабельність окремих оцінок, завдяки парності. Але bootstrap окремих рядків припускає незалежність об'єктів; за кластерної залежності слід ресемплювати незалежні групи, а за часової -- використовувати придатні блокові підходи.

Наступний код ілюструє *парний bootstrap* для різниці accuracy на незалежних синтетичних тестових об'єктах.


In [2]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_demo, y_demo = make_classification(
    n_samples=1100, n_features=12, n_informative=5,
    weights=[0.7, 0.3], flip_y=0.08, random_state=23
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_demo, y_demo, test_size=0.3, stratify=y_demo, random_state=23
)
models = {
    'baseline': DummyClassifier(strategy='most_frequent'),
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
}
predictions = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    predictions[name] = model.predict(X_te)

correct_A = (predictions['logistic'] == y_te).astype(float)
correct_B = (predictions['baseline'] == y_te).astype(float)
difference = correct_A - correct_B

rng = np.random.default_rng(2026)
boot = np.empty(3000)
for b in range(len(boot)):
    idx = rng.integers(0, len(difference), size=len(difference))
    boot[b] = difference[idx].mean()

print('Accuracy baseline:', round(correct_B.mean(), 3))
print('Accuracy logistic:', round(correct_A.mean(), 3))
print('Парна різниця accuracy:', round(difference.mean(), 3))
print('Ілюстративний percentile bootstrap interval:',
      np.round(np.quantile(boot, [0.025, 0.975]), 3))


Accuracy baseline: 0.694
Accuracy logistic: 0.879
Парна різниця accuracy: 0.185
Ілюстративний percentile bootstrap interval: [0.133 0.236]


### Межі інтерпретації

Цей percentile bootstrap-інтервал є *навчальною ілюстрацією*, а не універсальним методом інференції. Він враховує варіацію тестових об'єктів за фіксованих навчених моделей, але не включає варіацію, пов'язану з повторним відбором train-набору, навчанням чи налаштуванням. Для твердження про перевагу *алгоритму навчання* потрібен протокол, який відповідає ширшій оцінюваній величині.


## 12. Robustness: перевірка стійкості висновку

*Стійкість (robustness)* означає, що висновок не руйнується за обґрунтованих змін дослідницьких рішень. Перевірки можуть охоплювати:

- різні випадкові ініціалізації та seed;
- розумні альтернативи preprocessing і представлення ознак;
- альтернативні метрики, пороги, часові вікна;
- окремі групи, підпопуляції, періоди;
- шум, пропуски, зміну розподілу;
- зовнішній набір даних або незалежну реплікацію.

Важливо *відрізняти* заплановані перевірки стійкості від вибіркового оприлюднення лише тих налаштувань, які підтверджують бажаний результат.


## 13. Внутрішня, зовнішня, конструктна та статистична валідність

| Вид валідності | Ключове питання | Типова загроза |
|---|---|---|
| **Внутрішня** | Чи відповідає результат визначеній процедурі порівняння? | Leakage, confounding при порівнянні умов, нееквівалентні протоколи |
| **Зовнішня** | На які популяції, періоди й домени поширюється результат? | Вибірка з однієї платформи або одного часу |
| **Конструктна** | Чи вимірюють мітки та ознаки саме потрібне поняття? | «Бот» ототожнений з евристичним правилом |
| **Статистична** | Чи достатні дані та процедура для висновку про ефект? | Множинні порівняння, мала вибірка, ігнорування залежностей |

Термін *внутрішня валідність* у ML-оцінюванні слід застосовувати обережно: для причинних висновків він має спеціалізований зміст, який детально розглянемо в лекції 6 [@pearl2009causality; @hernan2020causal].


## 14. Researcher degrees of freedom і множинні спроби

Дослідник може змінювати вибірку, ознаки, метрику, seed, гіперпараметри, спосіб відкидання викидів і критерії відбору результатів. Якщо показати лише найкращу з багатьох спроб, опублікована оцінка може бути оптимістичною.

*Приклад.* Припустимо, кожна з 20 незалежних перевірок нульової гіпотези має імовірність хибнопозитивного результату 0,05. Імовірність принаймні одного хибнопозитивного результату дорівнює

$$
1-(1-0.05)^{20}\approx 0.642.
$$

Це *лише ілюстрація за припущення незалежності тестів*, а не універсальна формула для всіх ML-експериментів. Практичні запобіжники: заздалегідь зафіксований протокол, повне звітування про пошук, незалежний test, реплікація та аналіз чутливості.


## 15. Відтворюваність (reproducibility) і реплікованість (replicability)

*Обчислювальна відтворюваність* -- можливість повторити обчислення з тим самим кодом, даними та середовищем. *Реплікованість* -- отримати сумісний висновок в незалежному дослідженні (терміни можуть відрізнятися між дисциплінарними стандартами).

Мінімальний протокол: версії даних і коду, залежності Python, фіксовані seed, опис розбиття, всі гіперпараметри, журнал експериментів, артефакти оцінювання, опис обмежень доступу до даних. Для стохастичних обчислень seed підвищує контрольованість, але не гарантує бітової ідентичності між різними платформами та версіями бібліотек.

::: {.callout-note}
## Примітка
Відтворюваний *некоректний* експеримент залишається некоректним. Відтворюваність -- необхідна умова аудиту, а не достатня умова валідності.
:::


## 16. Матриця загроз валідності

```{mermaid}
%%{init: {'theme': 'base', 'themeVariables': {'fontSize':'18px','fontFamily':'Trebuchet MS, Verdana, Arial, sans-serif','primaryColor':'#E8F3FF','primaryBorderColor':'#3478A8','secondaryColor':'#FFF0C9','tertiaryColor':'#E7F6EC','lineColor':'#536878'}}}%%
flowchart LR
 D("Дані") --> L("Leakage / selection bias")
 M("Метод") --> H("Tuning / baseline bias")
 V("Валідація") --> G("Group / time mismatch")
 R("Результат") --> U("Uncertainty / multiplicity")
 L --> C("Аудит висновку")
 H --> C
 G --> C
 U --> C
```

Для кожної загрози варто зафіксувати: *джерело*, *можливий напрям спотворення*, *метод діагностики*, *коригувальну дію*, *залишкове обмеження*. Саме останній пункт переходить до розділу «Обмеження дослідження», а не зникає після побудови графіка.


## 17. Наскрізний кейс: раннє виявлення ботоподібної активності

*RQ.* Чи покращує поведінкова модель раннє виявлення ботоподібних облікових записів порівняно з простим baseline на *нових* облікових записах?

*Операціоналізація.* Ознаки доступні лише протягом перших $t$ годин; мітки отримані з незалежного джерела з описаними помилками; одиниця незалежності -- обліковий запис.

*Протокол.* Часове розбиття та групова ізоляція акаунтів; preprocessing лише на train; вибір гіперпараметрів на внутрішній валідації; одноразова фінальна перевірка на відкладеному часовому періоді.

*Метрики.* PR AUC, recall за обмеження false-positive rate, калібрування, час до виявлення; baseline -- просте правило й логістична регресія.

*Аудит.* Чутливість до часових вікон, нових груп користувачів, платформ, змін частки позитивного класу та якості міток.

*Допустиме твердження.* «За визначених джерел даних, часових меж і протоколу модель покращила зазначені показники відносно baseline».

*Недопустиме твердження.* «Модель довела причини ботоподібної поведінки» або «забезпечує таку саму якість на всіх платформах». Для причинних висновків потрібен інший дизайн [@peters2017elements].


## 18. Від результатів до наукового твердження

Формула академічно коректного підсумку:

> Для популяції **P**, на даних **D**, за протоколом **V**, модель **A** порівняно з **B** продемонструвала ефект **E** з оціненою невизначеністю **U**; результат стійкий до перевірок **S**, але обмежений умовами **L**.

Важливо розрізняти *результат експерименту*, *його інтерпретацію* і *узагальнене наукове твердження*. Переходи між ними повинні бути обґрунтовані, а не риторичні.


## 19. Зв'язок із лабораторною роботою 3

У лабораторній роботі 3 *«Валідація та аудит достовірності результатів машинного навчання»* аспірант застосує цю логіку до наскрізного ML/KDD-дослідження: перевірить розбиття, виявить можливі витоки, порівняє оцінки за альтернативними схемами, зафіксує невизначеність і складе матрицю загроз валідності.

Артефакт для дисертації: *аудит доказової сили ML-результату*, який доповнює RQ, baseline і протокол, сформовані в лабораторних роботах 1--2.


## 20. Підсумки

Валідний ML-експеримент починається до запуску алгоритму: з чіткої оцінюваної величини, визначення одиниці незалежності та протоколу порівняння. Вибір baseline, схеми валідації, метрики й способу обліку невизначеності визначає, *які саме свідчення* можна отримати. Перевірки стійкості, незалежний test та прозоре звітування зменшують ризик надмірних висновків, але не замінюють змістовної відповідності між даними й дослідницьким питанням.


## Питання для самоконтролю

1. Чим оцінювана величина (estimand) відрізняється від значення метрики на test?
2. Чому сильний baseline є частиною доказового дизайну?
3. Коли випадковий train/test split не відповідає цільовому сценарію?
4. Які механізми data leakage можливі навіть за формально правильного split?
5. Для чого потрібна nested CV?
6. Чому стандартне відхилення fold-метрик не завжди є довірчим інтервалом?
7. Чим парне порівняння моделей відрізняється від незалежного?
8. Які види валідності порушує невдала операціоналізація мітки?
9. Чому багаторазове використання test створює оптимізм оцінки?
10. Яка різниця між відтворюваністю експерименту та валідністю висновку?
11. Як сформулювати обмеження перенесення результату на інший домен?
12. Які перевірки стійкості потрібні саме для вашої дисертаційної задачі?


## Література

Основні концептуальні джерела: [@hastie2009elements; @bishop2006prml; @breiman2001two; @vapnik1998statistical]. Для розмежування прогностичного й причинного дизайну: [@pearl2009causality; @hernan2020causal].

::: {#refs}
:::
